In [17]:
import pandas as pd

df = pd.read_csv('christerhamp_gamla_runor_with_filenames.csv')

In [ ]:
# pip install google-genai Pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
df[df.runic.notna()].runic.tolist()[0]

'ᚼᚾᛁᛅᛋ\nᚠᚢᚦᚬᚱᚴ ᛬ ᚼᚾᛁᛅᛋ ᛬ ᛏᛒᛘᛚᛦ'

In [ ]:
"""
Synthetic OCR data pipeline for runic text — Gemini imagen backend.

Flow:
  1. Split long strings (\\n → always; space / ":" if still too long)
  2. Render each chunk → white-bg PIL image
  3. Send image + prompt to Gemini → stone/texture background, runes preserved
  4. Save (image, label) pairs + manifest.jsonl
"""

import io
import json
import time
import urllib.request
from pathlib import Path

from google import genai
from google.genai import types
from PIL import Image, ImageDraw, ImageFont
import pandas as pd


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

GEMINI_MODEL  = "gemini-3.1-flash-image-preview"
FONT_PATH     = "NotoSansRunic-Regular.ttf"
API_KEY = 'YOUR_GOOGLE_API_KEY'

# GitHub CDN — стабильный источник Noto Sans Runic
_FONT_URL = (
    "https://github.com/notofonts/NotoSansRunic/blob/main/fonts/ttf/unhinted/instance_ttf/NotoSansRunic-Regular.ttf"
)
FONT_SIZE     = 48
PADDING       = 24          # px вокруг текста
MAX_CHARS     = 30          # порог «длинной строки» для дополнительного сплита
N_VARIANTS    = 3           # сколько фонов генерировать на один чанк
RETRY_DELAY   = 5           # секунд между повторами при ошибке API
OUT_DIR       = Path("synth_data")
OUT_DIR.mkdir(exist_ok=True)



In [14]:


# ---------------------------------------------------------------------------
# Prompt
# ---------------------------------------------------------------------------

def build_prompt(rune_text: str) -> str:
    return f"""You are given an image of ancient runic inscription rendered in black on a white background.

Your task: KEEP THE RUNES EXACTLY AS THEY ARE and replace ONLY the white background with a photorealistic archaeological stone surface.

STRICT RULES:
- Do NOT alter, move, distort, rotate, or re-draw any rune character
- Do NOT add new rune characters or symbols
- Do NOT change the rune color — keep them dark/black
- ONLY change the background behind the runes
- The runes must remain perfectly legible after the background change


TARGET BACKGROUND STYLE:
Photorealistic archaeological macro photograph of an authentic Elder Futhark inscription carved into rough weathered grey granite.

Match this reference style exactly:
- extremely high contrast between runes and stone
- bright granite surface with dark deep rune grooves
- sharp chiselled angular cuts
- visible erosion and granular stone texture
- uneven ancient hand-cut geometry
- shallow engraved channels with crisp edges
- monochrome grey stone appearance
- tight horizontal crop focused only on inscription
- no cinematic fantasy atmosphere
- no glowing effects
- no stylization

STONE MATERIAL:
Ancient rough granite slab, coarse mineral grain, chipped edges, micro-fractures, weathered patina, tiny lichen traces, realistic erosion, visible hammer and chisel impact marks.

LIGHTING:
Hard directional side lighting from upper left, emphasizing groove shadows and stone relief. High local contrast. Clear rune readability. Bright illuminated granite surface.

CAMERA:
Extreme macro archaeological documentation photo, ultra sharp focus on carvings, shallow depth of field only outside inscription plane, high texture fidelity.

NEGATIVE:
blurred runes, fantasy runes, curved lines, Latin letters, smooth polished stone, soft engraving, modern typography, glowing symbols, cinematic fog, ornaments, watermark, extra symbols, unreadable carving, rounded shapes"""


# ---------------------------------------------------------------------------
# Step 1 — сплит длинных строк
# ---------------------------------------------------------------------------

def split_text(text: str, max_chars: int = MAX_CHARS) -> list[str]:
    """
    Правила (применяются последовательно):
      1. Всегда режем по \\n, если он есть.
      2. Если чанк всё ещё длиннее max_chars — режем по ':'
      3. Если всё ещё длинно — режем по пробелу.
    """
    # 1. по \n — всегда
    chunks = text.split("\n") if "\n" in text else [text]

    result = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        if len(chunk) <= max_chars:
            result.append(chunk)
            continue

        # 2. по ':'
        if ":" in chunk:
            sub = [s.strip() for s in chunk.split(":") if s.strip()]
        else:
            sub = [chunk]

        for part in sub:
            if len(part) <= max_chars:
                result.append(part)
                continue

            # 3. по пробелу (greedy packing)
            words = part.split(" ")
            line, line_len = [], 0
            for word in words:
                if line and line_len + 1 + len(word) > max_chars:
                    result.append(" ".join(line))
                    line, line_len = [word], len(word)
                else:
                    line.append(word)
                    line_len += (1 if line_len else 0) + len(word)
            if line:
                result.append(" ".join(line))

    return result or [text.strip()]


# ---------------------------------------------------------------------------
# Step 2 — рендер текста на белом фоне
# ---------------------------------------------------------------------------

_FONT_URL = "https://github.com/notofonts/NotoSansRunic/raw/refs/heads/main/fonts/ttf/unhinted/instance_ttf/NotoSansRunic-Regular.ttf"
def ensure_font(path: str = FONT_PATH) -> str:
    """
    Проверяет наличие шрифта. Если не найден — скачивает Noto Sans Runic.
    Возвращает итоговый путь к TTF.
    """
    import urllib.request
    p = Path(path)
    if p.exists():
        return str(p)

    # Только Linux-системные пути — seguisym и подобные не содержат рун
    system_candidates = [
        "/usr/share/fonts/truetype/noto/NotoSansRunic-Regular.ttf",
        "/usr/share/fonts/noto/NotoSansRunic-Regular.ttf",
    ]
    for candidate in system_candidates:
        if Path(candidate).exists():
            print(f"Font found: {candidate}")
            return candidate

    print(f"Downloading Noto Sans Runic → {p} ...")
    urllib.request.urlretrieve(_FONT_URL, p)
    print("Done.")
    return str(p)


def render_text_image(text: str) -> Image.Image:
    font  = ImageFont.truetype(ensure_font(), FONT_SIZE)
    dummy = Image.new("RGB", (1, 1))
    draw  = ImageDraw.Draw(dummy)
    bbox  = draw.textbbox((0, 0), text, font=font)
    w = bbox[2] - bbox[0] + PADDING * 2
    h = bbox[3] - bbox[1] + PADDING * 2

    img  = Image.new("RGB", (w, h), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    draw.text(
        (PADDING - bbox[0], PADDING - bbox[1]),
        text, font=font, fill=(0, 0, 0),
    )
    return img


# ---------------------------------------------------------------------------
# Step 3 — Gemini: меняем фон
# ---------------------------------------------------------------------------

def call_gemini(img: Image.Image, text: str, n: int = N_VARIANTS) -> list[Image.Image]:
    """
    Вызывает Gemini n раз, каждый раз получая один вариант фона.
    Возвращает список PIL Image.
    """
    prompt  = build_prompt(text)
    results = []

    for attempt in range(n):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=[prompt, img],
                config=types.GenerateContentConfig(
                    response_modalities=["IMAGE", "TEXT"],
                ),
            )
            for part in response.candidates[0].content.parts:
                if part.inline_data is not None:
                    results.append(part.as_image())
                    break
        except Exception as e:
            print(f"    ⚠ Gemini attempt {attempt + 1}/{n} failed: {e}")
            time.sleep(RETRY_DELAY)

    return results


# ---------------------------------------------------------------------------
# Step 4 — основной пайп
# ---------------------------------------------------------------------------

def process_dataframe(df: pd.DataFrame, runic_col: str = "runic") -> list[dict]:
    texts   = df[df[runic_col].notna()][runic_col].tolist()
    records = []
    img_idx = 0

    for row_i, text in enumerate(texts):
        chunks = split_text(text)
        print(f"\n[row {row_i + 1}/{len(texts)}] {repr(text)[:60]}")
        if len(chunks) > 1:
            print(f"  → split into {len(chunks)} chunks")

        for chunk_i, chunk in enumerate(chunks):
            print(f"  chunk {chunk_i}: {repr(chunk)}")

            # --- белый рендер (всегда сохраняем) ---
            white_img  = render_text_image(chunk)
            white_path = OUT_DIR / f"{img_idx:05d}_white.png"
            white_img.save(white_path)
            records.append({
                "image":      str(white_path),
                "text":       chunk,
                "bg":         "white",
                "source_row": row_i,
            })

            # --- augmented фоны ---
            aug_imgs = call_gemini(white_img, chunk)
            for aug_i, aug_img in enumerate(aug_imgs):
                aug_path = OUT_DIR / f"{img_idx:05d}_aug{aug_i}.png"
                aug_img.save(aug_path)
                records.append({
                    "image":      str(aug_path),
                    "text":       chunk,
                    "bg":         f"gemini_{aug_i}",
                    "source_row": row_i,
                })

            img_idx += 1

    manifest = OUT_DIR / "manifest.jsonl"
    with open(manifest, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"\n✓ {len(records)} записей → {manifest}")
    return records


# ---------------------------------------------------------------------------
# CLI / quick test
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # # --- тест сплита ---
    # samples = [
    #     "ᚼᚾᛁᛅᛋ\nᚠᚢᚦᚬᚱᚴ ᛬ ᚼᚾᛁᛅᛋ ᛬ ᛏᛒᛘᛚᛦ",
    #     "ᚹᛜᛏᛜᛜᚠᛁ ᚦᛏᛈᚷ",
    #     "ᚠᚢᚦᚬᚱᚴ:ᚼᚾᛁᛅᛋ:ᛏᛒᛘᛚᛦ ᚷᛖᛒᛟᛞ ᛚᚨᚷᚢᛊ",
    # ]
    # print("=== Split test ===")
    # for s in samples:
    #     print(f"IN:  {repr(s)}")
    #     print(f"OUT: {split_text(s)}\n")

    # # --- тест рендера (без Gemini) ---
    # img = render_text_image("ᚼᚾᛁᛅᛋ\nᚠᚢᚦᚬᚱᚴ ᛬ ᚼᚾᛁᛅᛋ ᛬ ᛏᛒᛘᛚᛦ")
    # img.save(OUT_DIR / "test_render.png")
    # print(f"Test render → {OUT_DIR}/test_render.png")

    # --- полный пайп ---
    df = df[(df.runic.notna()) & (df.transliteration.notna())].head(1)
    process_dataframe(df)


[row 1/1] 'ᛏᛁᚤᛁᛚᛚᛗᛁᚾᛁᛁᚱᛁᚾᚨᛚᛏᛏᚾ'
  chunk 0: 'ᛏᛁᚤᛁᛚᛚᛗᛁᚾᛁᛁᚱᛁᚾᚨᛚᛏᛏᚾ'
    ⚠ Gemini attempt 1/3 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-flash-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-flash-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-flash-image\nPlease retry in 41.942096224s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more abo

In [13]:
df[(df.runic.notna()) & (df.transliteration.notna())].head(1)

,stone_id,name,runic,transliteration,translation,page_url,image_url,image_filename
20,D RAÄ Silvberg 152,"D RAÄ Silvberg 152 - Dalarna: Sandvik, Silvberg",ᛏᛁᚤᛁᛚᛚᛗᛁᚾᛁᛁᚱᛁᚾᚨᛚᛏᛏᚾ,tiyillminiirinalttn,-,https://www.christerhamp.se/runor/gamla/d/draa...,https://www.christerhamp.se/runor/gamla/d/draa...,0020_D_RAÄ_Silvberg_152_D_RAÄ_Silvberg_152___D...


In [28]:
"""
Runic OCR synthetic data — render pipeline (no API calls).

Outputs:
  synth_data/single/  — one image per DataFrame row
  synth_data/paired/  — one image per consecutive pair of rows (stacked)
  synth_data/manifest_single.jsonl
  synth_data/manifest_paired.jsonl
"""

import json
import urllib.request
from pathlib import Path

import pandas as pd
from PIL import Image, ImageDraw, ImageFont


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

FONT_PATH  = "NotoSansRunic-Regular.ttf"
FONT_SIZE  = 48
PADDING    = 500      # px вокруг текста (одна строка)
LINE_GAP   = 500      # px между двумя строками в paired-режиме
MAX_CHARS  = 500     # порог для дополнительного сплита внутри строки

OUT_ROOT   = Path("synth_data")
DIR_SINGLE = OUT_ROOT / "single"
DIR_PAIRED = OUT_ROOT / "paired"

for d in (DIR_SINGLE, DIR_PAIRED):
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Font
# ---------------------------------------------------------------------------

_FONT_URL = (
    "https://github.com/notofonts/NotoSansRunic/blob/main/fonts/ttf/unhinted/instance_ttf/NotoSansRunic-Regular.ttf"
)

def ensure_font(path: str = FONT_PATH) -> str:
    p = Path(path)
    if p.exists():
        return str(p)
    for candidate in [
        "/usr/share/fonts/truetype/noto/NotoSansRunic-Regular.ttf",
        "/usr/share/fonts/noto/NotoSansRunic-Regular.ttf",
    ]:
        if Path(candidate).exists():
            return candidate
    print(f"Downloading Noto Sans Runic → {p} ...")
    urllib.request.urlretrieve(_FONT_URL, p)
    print("Done.")
    return str(p)

_font: ImageFont.FreeTypeFont | None = None

def get_font() -> ImageFont.FreeTypeFont:
    global _font
    if _font is None:
        _font = ImageFont.truetype(ensure_font(), FONT_SIZE)
    return _font


# ---------------------------------------------------------------------------
# Text splitting (unchanged)
# ---------------------------------------------------------------------------

def split_text(text: str, max_chars: int = MAX_CHARS) -> list[str]:
    """\\n → always split; then ':'; then space if still > max_chars."""
    chunks = text.split("\n") if "\n" in text else [text]
    result = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        if len(chunk) <= max_chars:
            result.append(chunk)
            continue
        sub = [s.strip() for s in chunk.split(":")] if ":" in chunk else [chunk]
        for part in sub:
            if not part:
                continue
            if len(part) <= max_chars:
                result.append(part)
                continue
            words, line, line_len = part.split(" "), [], 0
            for word in words:
                if line and line_len + 1 + len(word) > max_chars:
                    result.append(" ".join(line))
                    line, line_len = [word], len(word)
                else:
                    line.append(word)
                    line_len += (1 if line_len else 0) + len(word)
            if line:
                result.append(" ".join(line))
    return result or [text.strip()]


# ---------------------------------------------------------------------------
# Rendering helpers
# ---------------------------------------------------------------------------

def _text_bbox(draw: ImageDraw.ImageDraw, text: str):
    return draw.textbbox((0, 0), text, font=get_font())


def render_chunks(chunks: list[str]) -> Image.Image:
    """
    Рисует список чанков друг под другом на белом фоне.
    Используется и для single (1 строка DF → N чанков),
    и для paired (2 строки DF → N1+N2 чанков).
    """
    font  = get_font()
    dummy = Image.new("RGB", (1, 1))
    draw  = ImageDraw.Draw(dummy)

    bboxes = [_text_bbox(draw, c) for c in chunks]

    line_w  = [b[2] - b[0] for b in bboxes]
    line_h  = [b[3] - b[1] for b in bboxes]
    n       = len(chunks)

    total_w = max(line_w) + PADDING * 2
    total_h = sum(line_h) + PADDING * 2 + LINE_GAP * (n - 1)

    img  = Image.new("RGB", (total_w, total_h), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    y = PADDING
    for i, (chunk, bbox, h) in enumerate(zip(chunks, bboxes, line_h)):
        draw.text(
            (PADDING - bbox[0], y - bbox[1]),
            chunk, font=font, fill=(0, 0, 0),
        )
        y += h + (LINE_GAP if i < n - 1 else 0)

    return img


# ---------------------------------------------------------------------------
# Single-row images
# ---------------------------------------------------------------------------

def render_single(df: pd.DataFrame, runic_col: str = "runic_filtered") -> list[dict]:
    """
    Каждая строка DF → один PNG (все чанки строки на одном изображении).
    """
    records = []
    texts   = df[df[runic_col].notna()][runic_col].tolist()

    for idx, text in enumerate(texts):
        chunks = split_text(text)
        img    = render_chunks(chunks)
        path   = DIR_SINGLE / f"{idx:05d}.png"
        img.save(path)
        records.append({
            "image":  str(path),
            "text":   text,
            "chunks": chunks,
        })
        print(f"[single {idx+1}/{len(texts)}] {repr(text)[:50]}")

    manifest = OUT_ROOT / "manifest_single.jsonl"
    _save_manifest(records, manifest)
    return records


# ---------------------------------------------------------------------------
# Paired images (две строки DF на одном изображении)
# ---------------------------------------------------------------------------

def render_paired(df: pd.DataFrame, runic_col: str = "runic_filtered") -> list[dict]:
    """
    Берём строки попарно (0+1, 2+3, …).
    Если строк нечётное число — последняя идёт одна.
    """
    records = []
    texts   = df[df[runic_col].notna()][runic_col].tolist()
    pairs   = [texts[i:i+2] for i in range(0, len(texts), 2)]

    for idx, pair in enumerate(pairs):
        all_chunks = []
        labels     = []
        for text in pair:
            chunks = split_text(text)
            all_chunks.extend(chunks)
            labels.append(text)

        img  = render_chunks(all_chunks)
        path = DIR_PAIRED / f"{idx:05d}.png"
        img.save(path)
        records.append({
            "image":  str(path),
            "texts":  labels,          # исходные строки DF
            "chunks": all_chunks,      # все чанки объединённые
        })
        label_preview = " | ".join(repr(t)[:30] for t in labels)
        print(f"[paired {idx+1}/{len(pairs)}] {label_preview}")

    manifest = OUT_ROOT / "manifest_paired.jsonl"
    _save_manifest(records, manifest)
    return records


# ---------------------------------------------------------------------------
# Util
# ---------------------------------------------------------------------------

def _save_manifest(records: list[dict], path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  ✓ manifest → {path}  ({len(records)} records)")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # df = pd.read_parquet("your_data.parquet")

    # --- quick smoke test ---
    sample_data = {
        "runic": [
            "ᚼᚾᛁᛅᛋ\nᚠᚢᚦᚬᚱᚴ ᛬ ᚼᚾᛁᛅᛋ ᛬ ᛏᛒᛘᛚᛦ",
            "ᚹᛜᛏᛜᛜᚠᛁ ᚦᛏᛈᚷ",
            "ᚠᚢᚦᚬᚱᚴ:ᚼᚾᛁᛅᛋ:ᛏᛒᛘᛚᛦ",
            "ᚷᛖᛒᛟᛞ ᛚᚨᚷᚢᛊ ᚠᛖᚺᚢ",
            "ᛊᛏᚨᚾᛞᚨᚱᛞ ᚠᚢᚦᚬᚱᚴ",
        ]
    }
    # df = pd.read_csv('christerhamp_gamla_runor_with_filenames.csv')

    df = df[(df.runic_filtered.notna()) & (df.transliteration_filtered.notna())]
    
    df['runic'] = df.runic.astype(str).str.replace('×', '').str.replace('-','').str.replace(' ᛬','')

    print("\n=== SINGLE ===")
    render_single(df)

    print("\n=== PAIRED ===")
    render_paired(df)

    print("\nDone. Check synth_data/single/ and synth_data/paired/")


=== SINGLE ===
[single 1/249] 'ᛏᛁᚤᛁᛚᛚᛗᛁᚾᛁᛁᚱᛁᚾᚨᛚᛏᛏᚾ'
[single 2/249] 'ᛁᛁᚠᚢᛅᚦᚢᛘᛚᚱᛅᚦᛣᛁᛏᛁᚬ'
[single 3/249] 'ᛒᛁᛚ'
[single 4/249] 'ᛖᛖᛚᛁᛚ'
[single 5/249] 'ᚨᛚᚢ'
[single 6/249] ' ᛋᛁᚼᚠᚱᛁᚦᚱ ᛁ ᛆᛚᚠᛁᚿᛆ ᛚᛁᛏ ᚵᛂᚱᛆ ᛋᛐᛆᛁᚿ ᚼᛁᛋᛆᚿ ᚤᚡᛁᚱ ᛒᚢᛏᛆ
[single 7/249] 'ᚼᛆᛚᚠᚱᛁᚦᚱ'
[single 8/249] 'ᚼᛅᛁᛘᛅᛚᛏᚱ ᚼᛅᛁ ᛒᚬᛏᚢᛁᚦᚱ ᛘ ᚼᛅᛁᛘ ᚼᛅᛁ'
[single 9/249] 'ᚠᚢᚦᚭᚱᚴᚼᚾᛁᛅᛋᛏᛒᛘᛚᛦ  \nᚠᚢᛦᚴᛅᚾᛏ'
[single 10/249] ' ᚵᚢᚦ ᚿᛆᚦᛁ ᛁᚢᛆᚿᛋ ᛋᛁᛆᛚ ᛆᚠ ᚼᛁᛂᚱᚿᚢᛘ ᚼᛆᛦ ᛑᚤᛐᚱᛁᚱ ᛚᛁᛐᚢ ᛘ
[single 11/249] 'ᛚᛁᛏ  ᚴᛁᛂᚱᛅ  ᛋᛏᛅᛁᚾ  ᛂᚠᛁ'
[single 12/249] 'ᚱᛆᚦᛁᛅᚢᚦ ᚴᛁᛆᚱᚦᛁ ᛘᛁᚴ ᚠᚢᚱᛁᛦ ᛋᚢᛁ ᛋᛁᚾ ᛚᛁᚴᚾᚢᛁᛅᚱ ᚮ ᛘᛁᚴ ᛒ
[single 13/249] 'ᚴᚢᚦ'
[single 14/249] 'ᚦᚭᚱ'
[single 15/249] 'ᚮᛚᛚᛁ'
[single 16/249] 'ᛁ ᚱᛅᛋᛏᚢ ᛋᛏ'
[single 17/249] 'ᚾᛅᚠᚾᛁ  ᚱᛁᛋᚦᛁ  ᛋᛏᛁᚾ  ᚦᛅᛋᛁ ᛅᚠᛏᛁᛦ  ᛏᚢᚴᛅ  ᛒᚱᚢᚦᚢᚱ  ᛋᛁ 
[single 18/249] ' ᛘᛆᚱᛐᛂᚿ ᛘᛁᚴ ᚵᛁᛆᚱᚦᛂ'
[single 19/249] 'ᛒᛁᚢᛚᛅ ᛋᛅᛏᛁ ᛁᚠᛏᛁᛦ  ᛏᚢᚴᛅ ᛘᛅᚴ ᛋᛁᚾ'
[single 20/249] ' ᛘᛆᚱᛐᛂᚿ ᛘᛁᚴ ᚵᛁᛆᚱᚦᛂ'
[single 21/249] ' ᛘᛆᚱᛐᛂᚿ ᛘᛁᚴ ᚵᛁᛆᚱᚦᛂ'
[single 22/249] 'ᚴᚱᛁᛋᛐ ᚴᚱᛁᛋ'
[single 23/249] ' ᛘᛆᚱᛐᛂᚿ ᛘᛁᚴ ᚵᛁᛆᚱᚦᛂ '
[single 24/249] 'ᚭᛚᛆᚠ ᚱᛁ'
[single 25/249] 'ᚵᛦᛚᛆ'
[single 26/249] 'ᛏᚮᛔᛁ'
[single 27/249] 'ᛘᚿᚱᛁᚿ ᚭ

In [66]:
df[:50]

,stone_id,name,runic,transliteration,translation,page_url,image_url,image_filename,runic_filtered,transliteration_filtered
20,D RAÄ Silvberg 152,"D RAÄ Silvberg 152 - Dalarna: Sandvik, Silvberg",ᛏᛁᚤᛁᛚᛚᛗᛁᚾᛁᛁᚱᛁᚾᚨᛚᛏᛏᚾ,tiyillminiirinalttn,-,https://www.christerhamp.se/runor/gamla/d/draa...,https://www.christerhamp.se/runor/gamla/d/draa...,0020_D_RAÄ_Silvberg_152_D_RAÄ_Silvberg_152___D...,ᛏᛁᚤᛁᛚᛚᛗᛁᚾᛁᛁᚱᛁᚾᚨᛚᛏᛏᚾ,tiyillminiirinalttn
21,D RAÄ Vika 182,"D RAÄ Vika 182 - Dalarna: Nyhyttan, Vika",ᛁᛁᚠᚢᛅᚦᚢᛘᛚᚱᛅᚦᛣᛁᛏᛁᚬ,iifuaþumlraþRitio,-,https://www.christerhamp.se/runor/gamla/d/draa...,https://www.christerhamp.se/runor/gamla/d/draa...,0021_D_RAÄ_Vika_182_D_RAÄ_Vika_182___Dalarna__...,ᛁᛁᚠᚢᛅᚦᚢᛘᛚᚱᛅᚦᛣᛁᛏᛁᚬ,iifuaþumlraþRitio
86,G 90,"G 90 - Gotland: Sigdes, Burs",᛭ᛒᛁᛚ,÷ bil,-,https://www.christerhamp.se/runor/gamla/g/g90....,https://www.christerhamp.se/runor/gamla/g/g90s...,0086_G_90_G_90___Gotland__Sigdes__Burs.jpg,ᛒᛁᛚ,bil
164,G 204,"G 204 - Gotland: Djupbrunns, Hogrän",ᛖᛖᛚᛁᛚ,ee=lil,-,https://www.christerhamp.se/runor/gamla/g/g204...,https://www.christerhamp.se/runor/gamla/g/g204...,0164_G_204_G_204___Gotland__Djupbrunns__Hogrän...,ᛖᛖᛚᛁᛚ,ee=lil
165,G 205,"G 205 - Gotland: Djupbrunns, Hogrän",ᚨᛚᚢ,alu,-,https://www.christerhamp.se/runor/gamla/g/g205...,https://www.christerhamp.se/runor/gamla/g/g205...,0165_G_205_G_205___Gotland__Djupbrunns__Hogrän...,ᚨᛚᚢ,alu
180,G 231,G 231 - Gotland: Vallstena kyrka,᛭ ᛋᛁᚼᚠᚱᛁᚦᚱ ᛁ ᛆᛚᚠᛁᚿᛆ ᛚᛁᛏ ᚵᛂᚱᛆ ᛋᛐᛆᛁᚿ ᚼᛁᛋᛆᚿ ᚤᚡᛁᚱ ...,+ sihfriþr : i : alfina : lit : gera : stain :...,Sigfrid i Alvne lät göra denna sten över sin d...,https://www.christerhamp.se/runor/gamla/g/g231...,https://www.christerhamp.se/runor/gamla/g/g231...,0180_G_231_G_231___Gotland__Vallstena_kyrka.jpg,ᛋᛁᚼᚠᚱᛁᚦᚱ ᛁ ᛆᛚᚠᛁᚿᛆ ᛚᛁᛏ ᚵᛂᚱᛆ ᛋᛐᛆᛁᚿ ᚼᛁᛋᛆᚿ ᚤᚡᛁᚱ ᛒ...,+ sihfriþr : i : alfina : lit : gera : stain :...
183,G 236,G 236 - Gotland: Vallstena kyrka,ᚼᛆᛚᚠᚱᛁᚦᚱ,halfriþr,Hallfrid,https://www.christerhamp.se/runor/gamla/g/g236...,https://www.christerhamp.se/runor/gamla/g/g236...,0183_G_236_G_236___Gotland__Vallstena_kyrka.jpg,ᚼᛆᛚᚠᚱᛁᚦᚱ,halfriþr
187,G 247,G 247 - Gotland: Hejnums kyrka,ᚼᛅᛁᛘᛅᛚᛏᚱ ᚼᛅᛁ ᛒᚬᛏᚢᛁᚦᚱ ᛘ ᚼᛅᛁᛘ ᚼᛅᛁ,A: haimaltr B: hai C: botuiþr m haim hai,Haimaldr Ha... Botvid Haim... Hai...,https://www.christerhamp.se/runor/gamla/g/g247...,https://www.christerhamp.se/runor/gamla/g/g247...,0187_G_247_G_247___Gotland__Hejnums_kyrka.jpg,ᚼᛅᛁᛘᛅᛚᛏᚱ ᚼᛅᛁ ᛒᚬᛏᚢᛁᚦᚱ ᛘ ᚼᛅᛁᛘ ᚼᛅᛁ,A: haimaltr B: hai C: botuiþr m haim hai
200,G 281,"G 281 - Gotland: Västers, Boge",ᚠᚢᚦᚭᚱᚴᚼᚾᛁᛅᛋᛏᛒᛘᛚᛦ \nᚠᚢᛦᚴᛅᚾᛏ,fuþorkhniastbmlR fuRkant,-,https://www.christerhamp.se/runor/gamla/g/g281...,https://www.christerhamp.se/runor/gamla/g/g281...,0200_G_281_G_281___Gotland__Västers__Boge.jpg,ᚠᚢᚦᚭᚱᚴᚼᚾᛁᛅᛋᛏᛒᛘᛚᛦ \nᚠᚢᛦᚴᛅᚾᛏ,fuþorkhniastbmlR fuRkant
217,G 312,G 312 - Gotland: Halls kyrka,᛭ ᚵᚢᚦ ᚿᛆᚦᛁ ᛁᚢᛆᚿᛋ ᛋᛁᛆᛚ ᛆᚠ ᚼᛁᛂᚱᚿᚢᛘ ᚼᛆᛦ ᛑᚤᛐᚱᛁᚱ ᛚᛁ...,÷ guþ : naþi : iuans : sial : af : hiernum : h...,Gud nåde Johans själ från Hjärne [?]. Hans döt...,https://www.christerhamp.se/runor/gamla/g/g312...,https://www.christerhamp.se/runor/gamla/g/g312...,0217_G_312_G_312___Gotland__Halls_kyrka.jpg,ᚵᚢᚦ ᚿᛆᚦᛁ ᛁᚢᛆᚿᛋ ᛋᛁᛆᛚ ᛆᚠ ᚼᛁᛂᚱᚿᚢᛘ ᚼᛆᛦ ᛑᚤᛐᚱᛁᚱ ᛚᛁᛐ...,guþ : naþi : iuans : sial : af : hiernum : ha...


In [1]:
import pandas as pd
df = pd.read_csv('christerhamp_gamla_runor_with_filenames.csv')




df['runic_filtered'] = df.runic.astype(str).str.replace('×', '').str.replace('-','').str.replace(' ᛬','').str.replace(' ᛬','').str.replace('᛭','').str.replace('᛫','').str.replace('᛬','').str.replace('*','').str.replace(':','').str.replace('.','').str.replace("'",'').str.replace("X IHSVS",'').str.replace("/",'').str.replace("P",'').str.replace(" ·",'').str.replace("+",'').str.replace(" ECÐÐANÐESECANCANHV ELTELAN7HEALDAN     ORVM",'').str.replace(" ·",'').str.replace(" ·",'').str.replace(" ·",'')

df['transliteration_filtered'] = df.transliteration.astype(str).str.replace('×', '').str.replace('-','').str.replace('÷','').str.replace('×', '').str.replace('-','').str.replace(' ᛬','').str.replace(' ᛬','').str.replace('᛭','').str.replace('᛫','').str.replace('᛬','').str.replace('*','').str.replace(':','').str.replace('.','').str.replace("'",'').str.replace("X IHSVS",'').str.replace("/",'').str.replace("P",'').str.replace(" ·",'').str.replace("+",'').str.replace(" ECÐÐANÐESECANCANHV ELTELAN7HEALDAN     ORVM",'').str.replace(" ·",'').str.replace(" ·",'').str.replace(" ·",'')
df = df[(df.runic_filtered.notna()) & (df.transliteration_filtered.notna())]

20                                   ᛏᛁᚤᛁᛚᛚᛗᛁᚾᛁᛁᚱᛁᚾᚨᛚᛏᛏᚾ
21                                     ᛁᛁᚠᚢᛅᚦᚢᛘᛚᚱᛅᚦᛣᛁᛏᛁᚬ
86                                                   ᛒᛁᛚ
164                                                ᛖᛖᛚᛁᛚ
165                                                  ᚨᛚᚢ
                              ...                       
2878                              ᚵᛦᛆ ᛌᛅᚼᛁᚱ ᛆᛐᚦᚢ ᚴᛆᚴᚼᛅᛁᛘ
2916                                                 ᚠᚢᚦ
2920                                       ᛒᛆᛌᛘᛆᚱᚦᛅᚱᛒᛅᛁᚿ
2928     ᚠᛚᚨᚷᛞᚨᚠᚨᛁᚲᛁᚾᚨᛉᛁᛊᛏ  ᛗᚨᚷᛟᚱᛗᛁᚾᚨᛊᛊᛏᚨᛁᚾᚨ  ᛞᚨᛉᚠᚨᛁᚺᛁᛞᛟ
2939     ᚴᚱᛆᚿᛁ ᚴᛂᚱᚦᛁ ᚼᛆᛚᚠ ᚦᛁᛋᛁ  ᛁᚠᛏᛁᚱ  ᚴᛆᛚ  ᚠᛁ  ᛚᛆᚴᛆ ᛋᛁᚿ
Name: runic_filtered, Length: 249, dtype: str

In [38]:
df.runic.astype(str).str.replace('×', '').str.replace('-', '').str.replace('.', '')

20                                    ᛏᛁᚤᛁᛚᛚᛗᛁᚾᛁᛁᚱᛁᚾᚨᛚᛏᛏᚾ
21                                      ᛁᛁᚠᚢᛅᚦᚢᛘᛚᚱᛅᚦᛣᛁᛏᛁᚬ
86                                                   ᛭ᛒᛁᛚ
164                                                 ᛖᛖᛚᛁᛚ
165                                                   ᚨᛚᚢ
                              ...                        
2878                         ᚵᛦᛆ ᛬ ᛌᛅᚼᛁᚱ ᛬ ᛆᛐᚦᚢ ᛬ ᚴᛆᚴᚼᛅᛁᛘ
2916                                                  ᚠᚢᚦ
2920                                        ᛒᛆᛌᛘᛆᚱᚦᛅᚱᛒᛅᛁᚿ
2928      ᚠᛚᚨᚷᛞᚨᚠᚨᛁᚲᛁᚾᚨᛉᛁᛊᛏ  ᛗᚨᚷᛟᚱᛗᛁᚾᚨᛊᛊᛏᚨᛁᚾᚨ  ᛞᚨᛉᚠᚨᛁᚺᛁᛞᛟ
2939    ᚴᚱᛆᚿᛁ ᛬ ᚴᛂᚱᚦᛁ ᛬ ᚼᛆᛚᚠ ᛬ ᚦᛁᛋᛁ : ᛁᚠᛏᛁᚱ : ᚴᛆᛚ : ᚠᛁ...
Name: runic, Length: 249, dtype: str

In [74]:
df['runic_filtered'] = df.runic.astype(str).str.replace('×', '').str.replace('-','').str.replace(' ᛬','').str.replace(' ᛬','').str.replace('᛭','').str.replace('᛫','').str.replace('᛬','').str.replace('*','')

df['transliteration_filtered'] = df.transliteration.astype(str).str.replace('×', '').str.replace('-','').str.replace('÷','')